<a href="https://colab.research.google.com/github/LouiseHjorth/speciale-forecasting-hay/blob/main/Forecasting_speciale_Tidsserie_3_m%C3%A5neder_Small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indlæsning af data

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.metrics import mean_absolute_error

file_name = list(uploaded.keys())[0]
print("Fil:", file_name)

df = pd.read_excel(file_name)

# Data rensning

In [ ]:
month_map = {
    "jan": 1, "feb": 2, "mar": 3, "apr": 4,
    "may": 5, "jun": 6, "jul": 7, "aug": 8,
    "sep": 9, "oct": 10, "nov": 11, "dec": 12
}

month_clean = df["MonthName"].astype(str).str.strip().str.lower()

df["Month_num"] = (
    pd.to_numeric(month_clean, errors="coerce")
    .fillna(month_clean.map(month_map))
    .astype("Int64")
)

df["Size Name"] = df["Size Name"].astype(str).str.strip().str.lower()

df["Date"] = pd.to_datetime(
    df["Year"].astype(str) + "-" + df["Month_num"].astype(str) + "-01"
)

df.head()
df.tail()

# addtiplicative vs. additive

In [ ]:
df_small = df[df["Size Name"] == "small"].copy()
df_small = df_small.sort_values(["Year", "Month_num"])
df_small = df_small.set_index("Date")

ts_small = df_small["Sold Quantity"].asfreq("MS")

plt.figure(figsize=(12, 6))
plt.plot(ts_small)
plt.title("Sales - Size Small")
plt.xlabel("Date")
plt.ylabel("Sold Quantity")

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[2, 5, 8, 11]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Decomposition

In [ ]:

decomp_small = seasonal_decompose(ts_small, model="additive", period=12)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(12, 10),
    sharex=True
)

ax1.plot(decomp_small.observed)
ax1.set_ylabel("Observed")
ax1.set_title("Additive Seasonal Decomposition - Size Small")

ax2.plot(decomp_small.trend)
ax2.set_ylabel("Trend")

ax3.plot(decomp_small.seasonal)
ax3.set_ylabel("Seasonal")

ax4.plot(decomp_small.resid)
ax4.set_ylabel("Residuals")
ax4.set_xlabel("Date")

ax4.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[2, 5, 8, 11]))
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Traning/test split

In [ ]:
train = ts_small.loc[: "2025-11-01"]
test = ts_small.loc["2025-12-01":"2026-02-01"]

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(ts_small.index, ts_small.values)

ax.set_xlabel("Date")
ax.set_ylabel("Sold Quantity")
ax.set_title("Train/Test Split - Size Small")

ax.axvspan(
    test.index.min(),
    test.index.max(),
    color="#808080",
    alpha=0.2
)

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[2, 5, 8, 11]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Baseline models

# Predict historical mean

In [ ]:
historical_mean = np.mean(train)
print("Historical mean:", historical_mean)

mean_forecast = pd.Series(historical_mean, index=test.index)

mean_residuals = train - historical_mean
mean_std = mean_residuals.std()

mean_lower = mean_forecast - 1.96 * mean_std
mean_upper = mean_forecast + 1.96 * mean_std

comparison = pd.DataFrame({
    "Actual": test,
    "Mean Forecast": mean_forecast,
    "Lower 95%": mean_lower,
    "Upper 95%": mean_upper
})

comparison["Absolute Error"] = abs(
    comparison["Actual"] - comparison["Mean Forecast"]
)

comparison["Absolute Error %"] = abs(
    (comparison["Actual"] - comparison["Mean Forecast"]) /
    comparison["Actual"]
) * 100

mae_hist_mean = comparison["Absolute Error"].mean()

print(comparison.round(0))
print("\nMAE - Historical mean:", mae_hist_mean)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train.index, train.values, 'g-', label='Train')

ax.plot(
    test.index,
    test.values,
    'b-o',
    linewidth=2,
    markersize=8,
    markeredgecolor='black',
    label='Test'
)

ax.plot(
    mean_forecast.index,
    mean_forecast.values,
    'r--o',
    linewidth=2,
    markersize=8,
    markeredgecolor='black',
    label='Mean Forecast'
)

ax.fill_between(
    mean_forecast.index,
    mean_lower,
    mean_upper,
    color='red',
    alpha=0.2,
    label='95% CI'
)

ax.axvline(
    test.index.min(),
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='Split'
)

ax.set_xlabel('Date')
ax.set_ylabel('Sold Quantity')
ax.set_title('Historical Mean Forecast with 95% CI - Size Small')

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[2, 5, 8, 11]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))

ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Predict last year mean

In [ ]:
last_year_mean = train.iloc[-12:].mean()
print("Last year mean:", last_year_mean)

last_year_forecast = pd.Series(last_year_mean, index=test.index)

last_year_residuals = train - last_year_mean
last_year_std = last_year_residuals.std()

last_year_lower = last_year_forecast - 1.96 * last_year_std
last_year_upper = last_year_forecast + 1.96 * last_year_std

comparison_last_year = pd.DataFrame({
    "Actual": test,
    "Last Year Mean": last_year_forecast,
    "Lower 95%": last_year_lower,
    "Upper 95%": last_year_upper
})

comparison_last_year["Absolute Error"] = abs(
    comparison_last_year["Actual"] -
    comparison_last_year["Last Year Mean"]
)

comparison_last_year["Absolute Error %"] = abs(
    (comparison_last_year["Actual"] -
     comparison_last_year["Last Year Mean"]) /
    comparison_last_year["Actual"]
) * 100

mae_last_year = comparison_last_year["Absolute Error"].mean()

print(comparison_last_year.round(0))
print("\nMAE - Last year mean:", mae_last_year)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train.index, train.values, 'g-', label='Train')

ax.plot(
    test.index,
    test.values,
    'o',
    color='blue',
    markersize=12,
    markeredgecolor='black',
    label='Test'
)

ax.plot(
    last_year_forecast.index,
    last_year_forecast.values,
    'o',
    color='purple',
    markersize=12,
    markeredgecolor='black',
    label='Last Year Mean'
)

ax.fill_between(
    last_year_forecast.index,
    last_year_lower,
    last_year_upper,
    color='purple',
    alpha=0.2,
    label='95% CI'
)

ax.axvline(
    test.index.min(),
    color='black',
    linestyle='--',
    linewidth=1.5,
    label='Split'
)

ax.set_xlabel('Date')
ax.set_ylabel('Sold Quantity')
ax.set_title('Last Year Mean Forecast with 95% CI - Size Small')

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[2, 5, 8, 11]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))

ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


# Naive seasonal forecast

In [ ]:
naive_seasonal_forecast = pd.Series(index=test.index, dtype=float)

for date in test.index:
    naive_seasonal_forecast.loc[date] = train.loc[
        date - pd.DateOffset(years=1)
    ]

naive_fitted = train.shift(12)
naive_residuals = (train - naive_fitted).dropna()
naive_std = naive_residuals.std()

naive_lower = naive_seasonal_forecast - 1.96 * naive_std
naive_upper = naive_seasonal_forecast + 1.96 * naive_std

comparison_seasonal = pd.DataFrame({
    "Actual": test,
    "Naive Seasonal Forecast": naive_seasonal_forecast,
    "Lower 95%": naive_lower,
    "Upper 95%": naive_upper
})

comparison_seasonal["Absolute Error"] = abs(
    comparison_seasonal["Actual"] -
    comparison_seasonal["Naive Seasonal Forecast"]
)

comparison_seasonal["Absolute Error %"] = abs(
    (comparison_seasonal["Actual"] -
     comparison_seasonal["Naive Seasonal Forecast"]) /
    comparison_seasonal["Actual"]
) * 100

mae_naive_seasonal = comparison_seasonal["Absolute Error"].mean()

print(comparison_seasonal.round(0))
print("\nMAE - Naive seasonal:", mae_naive_seasonal)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train.index, train.values, 'g-', label='Train')

ax.plot(
    test.index,
    test.values,
    'o',
    color='blue',
    markersize=12,
    markeredgecolor='black',
    label='Test'
)

ax.plot(
    naive_seasonal_forecast.index,
    naive_seasonal_forecast.values,
    'o',
    color='brown',
    markersize=12,
    markeredgecolor='black',
    label='Naive Seasonal Forecast'
)

ax.fill_between(
    naive_seasonal_forecast.index,
    naive_lower,
    naive_upper,
    color='brown',
    alpha=0.2,
    label='95% CI'
)

ax.axvline(
    test.index.min(),
    color='black',
    linestyle='--',
    linewidth=1.5
)

ax.set_xlabel('Date')
ax.set_ylabel('Sold Quantity')
ax.set_title('Naive Seasonal Forecast with 95% CI - Size Small')

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[2, 5, 8, 11]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%Y"))

ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Comparison - Baseline

In [ ]:
mae_hist_mean = mean_absolute_error(test, mean_forecast)
mae_last_year = mean_absolute_error(test, last_year_forecast)
mae_naive_seasonal = mean_absolute_error(test, naive_seasonal_forecast)

fig, ax = plt.subplots(figsize=(8, 5))

x = ['Historical mean', 'Last year mean', 'Naive seasonal']
y = [mae_hist_mean, mae_last_year, mae_naive_seasonal]

ax.bar(x, y, width=0.4)

ax.set_xlabel('Baselines')
ax.set_ylabel('MAE')
ax.set_title('Baseline Comparison (MAE) - Size Small')
ax.set_ylim(0, max(y) * 1.1)

for index, value in enumerate(y):
    plt.text(
        x=index,
        y=value * 1.02,
        s=f"{value:.0f}",
        ha='center'
    )

plt.tight_layout()
plt.show()

# Testing for stationarity

In [ ]:
result = adfuller(ts_small.dropna())

print("ADF Statistic:", result[0])
print("p-value:", result[1])

fig = plt.figure(figsize=(12, 8))

ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
ax1.plot(ts_small, marker='o', linestyle='-')
ax1.set_title("sales")

ax2 = plt.subplot2grid((2, 2), (1, 0))
plot_acf(ts_small.dropna(), lags=24, ax=ax2)
ax2.set_title("ACF")

ax3 = plt.subplot2grid((2, 2), (1, 1))
plot_pacf(ts_small.dropna(), lags=24, ax=ax3)
ax3.set_title("PACF")

plt.tight_layout()
plt.show()

# ARIMA

In [ ]:

arima_model = ARIMA(train, order=(1, 0, 1))
arima_fit = arima_model.fit()

arima_pred = arima_fit.get_forecast(steps=len(test))
arima_forecast = arima_pred.predicted_mean
arima_ci = arima_pred.conf_int(alpha=0.05)

arima_lower = arima_ci.iloc[:, 0]
arima_upper = arima_ci.iloc[:, 1]

mae_arima = mean_absolute_error(test, arima_forecast)

arima_results = pd.DataFrame({
    "Actual": test,
    "ARIMA Forecast": arima_forecast,
    "Lower 95%": arima_lower,
    "Upper 95%": arima_upper
})

arima_results["Absolute Error"] = abs(
    arima_results["Actual"] - arima_results["ARIMA Forecast"]
)

arima_results["Absolute Error %"] = (
    arima_results["Absolute Error"] / arima_results["Actual"] * 100
)

print(arima_results.round(0))
print("ARIMA mae:", mae_arima)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train.index, train.values, 'g-', label='Train')

ax.plot(
    test.index,
    test.values,
    'o',
    color='blue',
    markersize=12,
    markeredgecolor='black',
    label='Test'
)

ax.plot(
    arima_forecast.index,
    arima_forecast.values,
    'o',
    color='red',
    markersize=12,
    markeredgecolor='black',
    label='ARIMA Forecast'
)

ax.fill_between(
    arima_ci.index,
    arima_lower,
    arima_upper,
    color='red',
    alpha=0.2,
    label='95% CI'
)

ax.axvline(test.index.min(), color='black', linestyle='--', linewidth=1.5)

ax.set_title("ARIMA Forecast with 95% CI")
ax.set_xlabel("Date")
ax.set_ylabel("Sold Quantity")
ax.legend()

plt.show()

# SARIMA

In [ ]:
sarima_model = SARIMAX(
    train,
    order=(1, 0, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_fit = sarima_model.fit()

sarima_pred = sarima_fit.get_forecast(steps=len(test))
sarima_forecast = sarima_pred.predicted_mean
sarima_ci = sarima_pred.conf_int(alpha=0.05)

sarima_lower = sarima_ci.iloc[:, 0]
sarima_upper = sarima_ci.iloc[:, 1]

mae_sarima = mean_absolute_error(test, sarima_forecast)

sarima_results = pd.DataFrame({
    "Actual": test,
    "SARIMA Forecast": sarima_forecast,
    "Lower 95%": sarima_lower,
    "Upper 95%": sarima_upper
})

sarima_results["Absolute Error"] = abs(
    sarima_results["Actual"] - sarima_results["SARIMA Forecast"]
)

sarima_results["Absolute Error %"] = (
    sarima_results["Absolute Error"] / sarima_results["Actual"] * 100
)

print(sarima_results.round(0))
print("SARIMA mae:", mae_sarima)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train.index, train.values, 'g-', label='Train')

ax.plot(
    test.index,
    test.values,
    'o',
    color='blue',
    markersize=12,
    markeredgecolor='black',
    label='Test'
)

ax.plot(
    sarima_forecast.index,
    sarima_forecast.values,
    'o',
    color='purple',
    markersize=12,
    markeredgecolor='black',
    label='SARIMA Forecast'
)

ax.fill_between(
    sarima_ci.index,
    sarima_lower,
    sarima_upper,
    color='purple',
    alpha=0.2,
    label='95% CI'
)

ax.axvline(test.index.min(), color='black', linestyle='--', linewidth=1.5)

ax.set_title("SARIMA Forecast with 95% CI")
ax.set_xlabel("Date")
ax.set_ylabel("Sold Quantity")
ax.legend()

plt.show()

# Holt winthers

In [ ]:
hw_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=12
).fit()

hw_forecast = hw_model.forecast(len(test))

hw_residual_std = hw_model.resid.std()

hw_lower = hw_forecast - 1.96 * hw_residual_std
hw_upper = hw_forecast + 1.96 * hw_residual_std

mae_hw = mean_absolute_error(test, hw_forecast)

hw_results = pd.DataFrame({
    "Actual": test,
    "Holt-Winters Forecast": hw_forecast,
    "Lower 95%": hw_lower,
    "Upper 95%": hw_upper
})

hw_results["Absolute Error"] = abs(
    hw_results["Actual"] - hw_results["Holt-Winters Forecast"]
)

hw_results["Absolute Error %"] = (
    hw_results["Absolute Error"] / hw_results["Actual"] * 100
)

print(hw_results.round(0))
print("Holt-Winters mae:", mae_hw)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(train.index, train.values, 'g-', label='Train')

ax.plot(
    test.index,
    test.values,
    'o',
    color='blue',
    markersize=12,
    markeredgecolor='black',
    label='Test'
)

ax.plot(
    hw_forecast.index,
    hw_forecast.values,
    'o',
    color='orange',
    markersize=12,
    markeredgecolor='black',
    label='Holt-Winters Forecast'
)

ax.fill_between(
    hw_forecast.index,
    hw_lower,
    hw_upper,
    color='orange',
    alpha=0.2,
    label='95% CI'
)

ax.axvline(test.index.min(), color='black', linestyle='--', linewidth=1.5)

ax.set_title("Holt-Winters Forecast with 95% CI")
ax.set_xlabel("Date")
ax.set_ylabel("Sold Quantity")
ax.legend()

plt.show()

results = []

model_1 = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=12
).fit()

forecast_1 = model_1.forecast(len(test))
std_1 = model_1.resid.std()
lower_1 = forecast_1 - 1.96 * std_1
upper_1 = forecast_1 + 1.96 * std_1

mae_1 = mean_absolute_error(test, forecast_1)
results.append(("HW add trend + add season", mae_1))

hw1_results = pd.DataFrame({
    "Actual": test,
    "Forecast": forecast_1,
    "Lower 95%": lower_1,
    "Upper 95%": upper_1
})

print("\nHW add trend + add season")
print(hw1_results.round(0))
print("MAE:", mae_1)

model_2 = ExponentialSmoothing(
    train,
    trend=None,
    seasonal="add",
    seasonal_periods=12
).fit()

forecast_2 = model_2.forecast(len(test))
std_2 = model_2.resid.std()
lower_2 = forecast_2 - 1.96 * std_2
upper_2 = forecast_2 + 1.96 * std_2

mae_2 = mean_absolute_error(test, forecast_2)
results.append(("HW no trend + add season", mae_2))

hw2_results = pd.DataFrame({
    "Actual": test,
    "Forecast": forecast_2,
    "Lower 95%": lower_2,
    "Upper 95%": upper_2
})

print("\nHW no trend + add season")
print(hw2_results.round(0))
print("MAE:", mae_2)

model_3 = ExponentialSmoothing(
    train,
    trend="add",
    seasonal=None
).fit()

forecast_3 = model_3.forecast(len(test))
std_3 = model_3.resid.std()
lower_3 = forecast_3 - 1.96 * std_3
upper_3 = forecast_3 + 1.96 * std_3

mae_3 = mean_absolute_error(test, forecast_3)
results.append(("HW add trend only", mae_3))

hw3_results = pd.DataFrame({
    "Actual": test,
    "Forecast": forecast_3,
    "Lower 95%": lower_3,
    "Upper 95%": upper_3
})

print("\nHW add trend only")
print(hw3_results.round(0))
print("MAE:", mae_3)

model_4 = ExponentialSmoothing(
    train,
    trend=None,
    seasonal=None
).fit()

forecast_4 = model_4.forecast(len(test))
std_4 = model_4.resid.std()
lower_4 = forecast_4 - 1.96 * std_4
upper_4 = forecast_4 + 1.96 * std_4

mae_4 = mean_absolute_error(test, forecast_4)
results.append(("HW simple", mae_4))

hw4_results = pd.DataFrame({
    "Actual": test,
    "Forecast": forecast_4,
    "Lower 95%": lower_4,
    "Upper 95%": upper_4
})

print("\nHW simple")
print(hw4_results.round(0))
print("MAE:", mae_4)


# Evaluering

In [ ]:

forecast_summary = pd.DataFrame({
    "Actual": test
})

forecast_summary["Historical mean Forecast"] = mean_forecast
forecast_summary["Historical mean Lower 95%"] = mean_lower
forecast_summary["Historical mean Upper 95%"] = mean_upper

forecast_summary["Last year mean Forecast"] = last_year_forecast
forecast_summary["Last year mean Lower 95%"] = last_year_lower
forecast_summary["Last year mean Upper 95%"] = last_year_upper

forecast_summary["Naive seasonal Forecast"] = naive_seasonal_forecast
forecast_summary["Naive seasonal Lower 95%"] = naive_lower
forecast_summary["Naive seasonal Upper 95%"] = naive_upper

forecast_summary["ARIMA Forecast"] = arima_forecast
forecast_summary["ARIMA Lower 95%"] = arima_lower
forecast_summary["ARIMA Upper 95%"] = arima_upper

forecast_summary["SARIMA Forecast"] = sarima_forecast
forecast_summary["SARIMA Lower 95%"] = sarima_lower
forecast_summary["SARIMA Upper 95%"] = sarima_upper

forecast_summary["HW add trend + add season Forecast"] = forecast_1
forecast_summary["HW add trend + add season Lower 95%"] = lower_1
forecast_summary["HW add trend + add season Upper 95%"] = upper_1

forecast_summary["HW no trend + add season Forecast"] = forecast_2
forecast_summary["HW no trend + add season Lower 95%"] = lower_2
forecast_summary["HW no trend + add season Upper 95%"] = upper_2

forecast_summary["HW add trend only Forecast"] = forecast_3
forecast_summary["HW add trend only Lower 95%"] = lower_3
forecast_summary["HW add trend only Upper 95%"] = upper_3

forecast_summary["HW simple Forecast"] = forecast_4
forecast_summary["HW simple Lower 95%"] = lower_4
forecast_summary["HW simple Upper 95%"] = upper_4

forecast_summary = forecast_summary.round(0).astype(int)

print("\nSamlet forecasttabel med 95% CI")
print(forecast_summary)

mae_summary = pd.DataFrame({
    "Model": [
        "Historical mean",
        "Last year mean",
        "Naive seasonal",
        "ARIMA",
        "SARIMA",
        "HW add trend + add season",
        "HW no trend + add season",
        "HW add trend only",
        "HW simple"
    ],
    "MAE": [
        mae_hist_mean,
        mae_last_year,
        mae_naive_seasonal,
        mae_arima,
        mae_sarima,
        mae_1,
        mae_2,
        mae_3,
        mae_4
    ]
})

mae_summary["MAE"] = mae_summary["MAE"].round(0).astype(int)
mae_summary = mae_summary.sort_values("MAE").reset_index(drop=True)

print("\nMAE comparison")
print(mae_summary)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nHele forecasttabellen med CI")
print(forecast_summary.to_string())

forecast_summary.to_excel("forecast_summary_with_confidence_intervals_small.xlsx")
mae_summary.to_excel("mae_summary_small.xlsx", index=False)